##Read Silver Tables

In [0]:
customers_df = spark.table("workspace.default.silver_customers")
orders_df = spark.table("workspace.default.silver_orders")
items_df = spark.table("workspace.default.silver_items")
payments_df = spark.table("workspace.default.silver_payments")
products_df = spark.table("workspace.default.silver_products")

##Gold Table 1 - Customer 360

In [0]:
from pyspark.sql.functions import (
    sum,
    avg,
    countDistinct,
    min,
    max,
    datediff,
    current_date
)

customer360_df = (
    customers_df
    .join(orders_df, "customer_id", "left")
    .join(payments_df, "order_id", "left")
    .groupBy(
        "customer_id",
        "customer_city",
        "customer_state"
    )
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("payment_value").alias("total_spent"),
        avg("payment_value").alias("avg_order_value"),
        min("order_purchase_timestamp").alias("first_purchase"),
        max("order_purchase_timestamp").alias("last_purchase")
    )
)

customer360_df = customer360_df.withColumn(
    "customer_lifetime_days",
    datediff(current_date(), customer360_df.last_purchase)
)

In [0]:
customer360_df.show(10, truncate=False)

+-----------+--------------+--------------+------------+-----------------+------------------+-------------------+-------------------+----------------------+
|customer_id|customer_city |customer_state|total_orders|total_spent      |avg_order_value   |first_purchase     |last_purchase      |customer_lifetime_days|
+-----------+--------------+--------------+------------+-----------------+------------------+-------------------+-------------------+----------------------+
|CUST_003610|belem         |PA            |1           |162.18           |162.18            |2022-07-02 08:17:00|2022-07-02 08:17:00|1469                  |
|CUST_004929|porto velho   |RO            |4           |6439.73          |1609.9325         |2021-01-19 16:35:00|2022-12-31 18:38:00|1287                  |
|CUST_007914|sao paulo     |SP            |5           |9997.939999999999|1428.2771428571427|2021-03-14 05:58:00|2023-04-20 09:28:00|1177                  |
|CUST_010325|belem         |PA            |1           |12

In [0]:
customer360_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("workspace.default.gold_customer360")

In [0]:
    %sql

SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,bronze_customers,false
default,bronze_items,false
default,bronze_orders,false
default,bronze_payments,false
default,bronze_products,false
default,gold_customer360,false
default,gold_delivery_performance,false
default,gold_monthly_sales,false
default,gold_payment_summary,false
default,gold_product_performance,false


##Gold Table 2 — Monthly Sales

In [0]:
from pyspark.sql.functions import date_format, sum, countDistinct, avg

monthly_sales_df = (
    orders_df
    .join(payments_df, "order_id")
    .groupBy(
        date_format("order_purchase_timestamp", "yyyy-MM").alias("sales_month")
    )
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("payment_value").alias("total_revenue"),
        avg("payment_value").alias("average_order_value")
    )
    .orderBy("sales_month")
)

In [0]:
monthly_sales_df.show()

+-----------+------------+------------------+-------------------+
|sales_month|total_orders|     total_revenue|average_order_value|
+-----------+------------+------------------+-------------------+
|    2021-01|        1684| 2517124.730000002| 1294.8172479423877|
|    2021-02|        1519| 2292457.679999997| 1320.5401382488462|
|    2021-03|        1779|2634809.7499999953|  1290.308398628793|
|    2021-04|        1709|2621228.9199999943| 1338.7277425944812|
|    2021-05|        1722| 2632810.160000004|  1311.813731938218|
|    2021-06|        1654|2523478.2900000014| 1326.7498895899062|
|    2021-07|        1730|2622612.1100000017| 1313.9339228456922|
|    2021-08|        1657| 2523118.710000002| 1318.9329377940417|
|    2021-09|        1672|2587478.9000000022| 1346.2429240374622|
|    2021-10|        1735| 2646900.279999996| 1332.7795971802598|
|    2021-11|        1680|2545607.3000000007| 1326.5280354351228|
|    2021-12|        1734|2631296.3900000053| 1329.6090904497248|
|    2022-

In [0]:
monthly_sales_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("workspace.default.gold_monthly_sales")

##Gold Table 3 — Product Performance

In [0]:
silver_items_df = spark.table("workspace.default.silver_items")

silver_products_df = spark.table("workspace.default.silver_products")

In [0]:
from pyspark.sql.functions import (
    count,
    sum as spark_sum,
    avg,
    round,
    col
)

# =====================================================
# Product Performance
# =====================================================

product_performance_df = (

    silver_items_df

    .join(
        silver_products_df,
        "product_id"
    )

    .groupBy("product_category_name")

    .agg(

        count("*").alias("units_sold"),

        round(
            spark_sum("price"),
            2
        ).alias("total_revenue"),

        round(
            avg("price"),
            2
        ).alias("average_price")

    )

)

# =====================================================
# Revenue Contribution %
# =====================================================

total_revenue = product_performance_df.agg(
    spark_sum("total_revenue")
).collect()[0][0]

product_performance_df = (

    product_performance_df

    .withColumn(

        "revenue_percentage",

        round(

            (col("total_revenue") / total_revenue) * 100,

            2

        )

    )

    .orderBy(
        col("total_revenue").desc()
    )

)

product_performance_df.show(truncate=False)

+---------------------+----------+-------------+-------------+------------------+
|product_category_name|units_sold|total_revenue|average_price|revenue_percentage|
+---------------------+----------+-------------+-------------+------------------+
|garden               |6661      |8387753.24   |1259.23      |7.72              |
|furniture            |6429      |8123810.03   |1263.62      |7.48              |
|toys                 |6197      |7845704.42   |1266.05      |7.22              |
|books                |6349      |7835157.77   |1234.08      |7.21              |
|health               |6084      |7558586.97   |1242.37      |6.96              |
|automotive           |5883      |7482324.01   |1271.86      |6.89              |
|electronics          |5813      |7251155.07   |1247.4       |6.68              |
|music                |5560      |7039716.52   |1266.14      |6.48              |
|fashion              |5462      |6931312.2    |1269.01      |6.38              |
|beauty         

In [0]:
product_performance_df.show(10)

+---------------------+----------+-----------------+------------------+
|product_category_name|units_sold|    total_revenue|     average_price|
+---------------------+----------+-----------------+------------------+
|               garden|      6661| 8387753.23999998| 1259.233334334181|
|            furniture|      6429|8123810.030000023|1263.6195411417052|
|                 toys|      6197| 7845704.42000002| 1266.048801032761|
|                books|      6349|7835157.769999998| 1234.077456292329|
|               health|      6084|7558586.970000016| 1242.371296844184|
|           automotive|      5883|7482324.009999993|1271.8551776304596|
|          electronics|      5813|7251155.070000001|1247.4032461723725|
|                music|      5560|7039716.519999979|1266.1360647481977|
|              fashion|      5462|6931312.199999991|1269.0062614426934|
|               beauty|      5475|6845945.019999977| 1250.400916894973|
+---------------------+----------+-----------------+------------

In [0]:
product_performance_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("workspace.default.gold_product_performance")

##Gold Table 4 — Payment Summary

In [0]:
from pyspark.sql.functions import sum, count

payment_summary_df = (
    payments_df
    .groupBy("payment_type")
    .agg(
        count("*").alias("number_of_payments"),
        sum("payment_value").alias("total_amount")
    )
    .orderBy("total_amount", ascending=False)
)

In [0]:
payment_summary_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("workspace.default.gold_payment_summary")

##Gold Table 5 — Delivery Performance

In [0]:
from pyspark.sql.functions import avg, min, max, count

delivery_performance_df = (
    orders_df
    .groupBy("order_status")
    .agg(
        count("*").alias("total_orders"),
        avg("delivery_days").alias("avg_delivery_days"),
        min("delivery_days").alias("fastest_delivery"),
        max("delivery_days").alias("slowest_delivery")
    )
)

In [0]:
delivery_performance_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("workspace.default.gold_delivery_performance")

In [0]:
%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,bronze_customers,false
default,bronze_items,false
default,bronze_orders,false
default,bronze_payments,false
default,bronze_products,false
default,gold_customer360,false
default,gold_delivery_performance,false
default,gold_monthly_sales,false
default,gold_payment_summary,false
default,gold_product_performance,false


In [0]:
# Validate Gold Tables

spark.table("workspace.default.gold_customer360").show(10)

spark.table("workspace.default.gold_monthly_sales").show(10)

spark.table("workspace.default.gold_product_performance").show(10)

spark.table("workspace.default.gold_payment_summary").show(10)

spark.table("workspace.default.gold_delivery_performance").show(10)

+-----------+------------+------------------+------------------+-------------------+
|customer_id|total_orders|       total_spent|   avg_order_value|      last_purchase|
+-----------+------------+------------------+------------------+-------------------+
|CUST_009799|           7|           6710.19| 958.5985714285714|2023-06-08 22:16:00|
|CUST_012836|           6|           4467.66|            744.61|2023-05-14 10:07:00|
|CUST_000205|           5|           9554.42|          1910.884|2023-05-14 17:02:00|
|CUST_007474|           2|           2744.19|          1372.095|2023-04-06 09:00:00|
|CUST_002407|           6|2649.6800000000003| 441.6133333333334|2022-09-11 22:19:00|
|CUST_006355|           9|13948.220000000001|1549.8022222222223|2023-02-11 03:15:00|
|CUST_007508|           8|          14043.06|         1755.3825|2023-06-20 16:52:00|
|CUST_006851|           5|3814.5200000000004| 762.9040000000001|2022-10-21 14:22:00|
|CUST_005504|           5|           9702.16|          1940.432|2

##Create the KPI DataFrame


In [0]:
from pyspark.sql.functions import (
    sum,
    avg,
    countDistinct,
    round,
    format_number
)

kpi_df = (
    orders_df
    .join(payments_df, "order_id")
    .agg(
        format_number(sum("payment_value"), 2).alias("total_revenue"),
        countDistinct("customer_id").alias("total_customers"),
        countDistinct("order_id").alias("total_orders"),
        round(avg("payment_value"), 2).alias("average_order_value"),
        round(avg("delivery_days"), 2).alias("average_delivery_days")
    )
)

In [0]:
kpi_df.show()

+-------------+---------------+------------+-------------------+---------------------+
|total_revenue|total_customers|total_orders|average_order_value|average_delivery_days|
+-------------+---------------+------------+-------------------+---------------------+
|76,046,455.09|          14481|       50000|            1325.13|                 10.1|
+-------------+---------------+------------+-------------------+---------------------+



In [0]:
kpi_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("workspace.default.gold_kpi_dashboard")